In [ ]:
# ============================================================
# CELL 6 — FIRST CELL OF A **BRAND NEW** COLAB NOTEBOOK.
#
# Before running: Runtime > Change runtime type > T4 GPU.
# Also upload handoff.json (folder icon on the left, the upload arrow at the top).
#
# This installs NeMo into a clean environment. Nothing else is installed, so
# there is no whisper/pyannote/numba conflict to resolve.
# Takes 5-10 minutes and prints many warnings. Warnings are fine.
# When it finishes: Runtime > Restart session, then run CELL 7.
# ============================================================

!apt-get -qq update > /dev/null && apt-get -qq install -y libsndfile1 ffmpeg > /dev/null
!pip install -q Cython packaging
!pip install -q "nemo_toolkit[asr]"

print()
print("=" * 60)
print("NeMo installed. Do NOT upgrade numba - NeMo pins what it needs.")
print("NEXT: Runtime > Restart session, then run CELL 7.")
print("=" * 60)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# CELL 12 — NOTEBOOK B (the NeMo one). base.en + Sortformer.
#
# Order in a fresh notebook:
#   Runtime > Change runtime type > T4 GPU
#   1. paste PASTE_6_new_notebook.py, run          (installs NeMo, 5-10 min)
#   2. Runtime > Restart session
#   3. upload handoff_base.json  (folder icon > upload arrow)
#   4. paste this cell, run                        (about 3 min)
#
# Sortformer only. No Parakeet, so the crash that needed a batched call is not
# in play here.
# ============================================================

import os, sys, json, time, gc, threading, subprocess, platform
import torch, psutil, soundfile as sf, pandas as pd

if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Wajeeha-Kamran/emr-assistant-backend.git",
                    "/content/repo"], check=True)
os.chdir("/content/repo")
sys.path.insert(0, os.getcwd())

assert os.path.exists("/content/handoff_base.json"), (
    "handoff_base.json is not uploaded. Folder icon on the left > upload arrow.")

from scripts.evaluate_accuracy import (
    parse_scripts, normalise, strip_numerics,
    word_error_rate, speaker_accuracy, SCRIPTS_MD,
)
scripts = parse_scripts(SCRIPTS_MD)

with open("/content/handoff_base.json") as f:
    HANDOFF = json.load(f)
AUDIO_DIR = HANDOFF["audio_dir"]

HARDWARE = {
    "platform": platform.platform(), "python": platform.python_version(),
    "torch": torch.__version__,
    "cpu_cores_physical": psutil.cpu_count(logical=False),
    "cpu_cores_logical": psutil.cpu_count(logical=True),
    "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
                   if torch.cuda.is_available() else None,
    "cuda": torch.version.cuda,
}
assert HARDWARE["gpu"], "No GPU. Runtime > Change runtime type > T4 GPU."
print(json.dumps(HARDWARE, indent=2))


class Measured:
    def __init__(self, label):
        self.label = label
        self.stats = {}

    def __enter__(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        self._proc = psutil.Process()
        self._cpu0 = sum(self._proc.cpu_times()[:2])
        self._rss_peak = self._proc.memory_info().rss
        self._stop = threading.Event()

        def sample():
            while not self._stop.wait(0.25):
                self._rss_peak = max(self._rss_peak, self._proc.memory_info().rss)

        self._sampler = threading.Thread(target=sample, daemon=True)
        self._sampler.start()
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        self.stats["wall_s"] = round(time.time() - self._t0, 2)
        self._stop.set()
        self._sampler.join(timeout=1)
        self.stats["cpu_s"] = round(sum(self._proc.cpu_times()[:2]) - self._cpu0, 2)
        self.stats["ram_peak_gb"] = round(self._rss_peak / 1e9, 2)
        self.stats["vram_peak_gb"] = (
            round(torch.cuda.max_memory_allocated() / 1e9, 2)
            if torch.cuda.is_available() else None)
        return False


def assign_roles(segments):
    """Label the more question-asking cluster DOCTOR. Mirrors DiarizationService."""
    counts = {}
    for s in segments:
        counts[s["speaker"]] = counts.get(s["speaker"], 0) + s["text"].count("?")
    if not counts:
        return segments
    doctor = max(counts, key=counts.get)
    for s in segments:
        s["speaker_role"] = "DOCTOR" if s["speaker"] == doctor else "PATIENT"
    return segments


def words_to_turns(words, spans):
    def speaker_at(t):
        for start, end, spk in spans:
            if start <= t <= end:
                return spk
        return min(spans, key=lambda s: min(abs(s[0] - t), abs(s[1] - t)))[2] if spans else "A"

    merged, current = [], None
    for w in words:
        spk = speaker_at((w["start"] + w["end"]) / 2)
        if current is None or current["speaker"] != spk:
            if current:
                merged.append(current)
            current = {"speaker": spk, "text": w["word"]}
        else:
            current["text"] += w["word"]
    if current:
        merged.append(current)
    return assign_roles(merged)


def score(n, segments):
    ref_words, ref_spk = [], []
    for speaker, text in scripts[n]:
        w = normalise(text)
        ref_words.extend(w)
        ref_spk.extend([speaker] * len(w))
    hyp_words, hyp_spk = [], []
    for seg in segments:
        w = normalise(seg["text"])
        hyp_words.extend(w)
        hyp_spk.extend([seg["speaker_role"]] * len(w))
    wacc = max(0.0, 1 - word_error_rate(
        strip_numerics(ref_words), strip_numerics(hyp_words))) * 100
    correct, total = speaker_accuracy(ref_words, ref_spk, hyp_words, hyp_spk)
    return wacc, (correct / total * 100) if total else 0.0


_SF = {}

def mono16k(wav):
    if wav in _SF:
        return _SF[wav]
    audio, sr = sf.read(wav)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        import librosa
        audio = librosa.resample(audio.astype("float32"), orig_sr=sr, target_sr=16000)
    os.makedirs("/content/sf16k", exist_ok=True)
    out = "/content/sf16k/" + os.path.basename(wav)
    sf.write(out, audio, 16000, subtype="PCM_16")
    _SF[wav] = out
    return out


# ------------------------------------------------------------------- run
LABEL = "5. base.en + sortformer"
DETAIL = "/content/base_sortformer_detail.csv"

print(f"\n=== {LABEL} ===")
from nemo.collections.asr.models import SortformerEncLabelModel

with Measured("load") as load_sf:
    sortformer = SortformerEncLabelModel.from_pretrained("nvidia/diar_sortformer_4spk-v1")
    sortformer.eval()
    if torch.cuda.is_available():
        sortformer = sortformer.cuda()
print(f"  load {load_sf.stats['wall_s']}s, VRAM {load_sf.stats['vram_peak_gb']}GB")

rows = []
for n in sorted(scripts):
    d = HANDOFF["scripts"][str(n)]

    with Measured(f"sf{n}") as m:
        pred = sortformer.diarize(audio=[mono16k(f"{AUDIO_DIR}/consult_{n}.wav")],
                                  batch_size=1)

    raw = pred[0] if isinstance(pred, (list, tuple)) and pred else pred
    spans = []
    for s in raw:
        if isinstance(s, str):
            p = s.replace(",", " ").split()
            spans.append((float(p[0]), float(p[1]), str(p[2])))
        elif isinstance(s, (list, tuple)) and len(s) >= 3:
            spans.append((float(s[0]), float(s[1]), str(s[2])))
        else:
            raise RuntimeError(f"unexpected Sortformer segment: {type(s)} {s!r}")

    segments = words_to_turns(d["whisper_base_words"], spans)
    wacc, spk = score(n, segments)

    w = d["whisper_base_stats"]
    # Stages run in sequence: time and CPU add up, peak memory is the larger
    # stage rather than the sum, because the two models are not co-resident.
    infer = round(w["wall_s"] + m.stats["wall_s"], 2)
    rows.append({
        "run": LABEL, "audio_set": os.path.basename(AUDIO_DIR), "script": n,
        "word_acc": round(wacc, 1), "speaker_acc": round(spk, 1),
        "segments": len(segments), "ref_turns": len(scripts[n]),
        "audio_s": round(d["audio_s"], 1), "infer_s": infer,
        "realtime_x": round(infer / d["audio_s"], 2) if d["audio_s"] else None,
        "cpu_s": round(w["cpu_s"] + m.stats["cpu_s"], 2),
        "vram_peak_gb": round(max(w["vram_peak_gb"] or 0,
                                  m.stats["vram_peak_gb"] or 0), 2),
        "ram_peak_gb": round(max(w["ram_peak_gb"], m.stats["ram_peak_gb"]), 2),
        "asr_load_s": w.get("wall_s"),
        "diar_load_s": load_sf.stats["wall_s"],
        "diar_load_vram_gb": load_sf.stats["vram_peak_gb"],
    })
    pd.DataFrame(rows).to_csv(DETAIL, index=False)   # rewritten after every script
    print(f"  script {n}: word {wacc:.1f}%  speaker {spk:.1f}%  "
          f"{len(segments)}/{len(scripts[n])} turns  [saved]")

detail = pd.DataFrame(rows)
print()
print("=" * 62)
print(f"MEAN  word {detail.word_acc.mean():.1f}%   "
      f"speaker {detail.speaker_acc.mean():.1f}%   "
      f"infer {detail.infer_s.mean():.1f}s   "
      f"peak VRAM {detail.vram_peak_gb.max()}GB   "
      f"peak RAM {detail.ram_peak_gb.max()}GB")
print("=" * 62)
print("\nExpected from the earlier runs: word ~88%, speaker ~99.9%, VRAM ~3GB.")
print("If it lands far from that, tell me before it goes in the report.")

with open("/content/base_sortformer_hardware.json", "w") as f:
    json.dump(HARDWARE, f, indent=2)
print("\nDownload base_sortformer_detail.csv and send it to me.")

{
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "cpu_cores_physical": 1,
  "cpu_cores_logical": 2,
  "ram_total_gb": 13.6,
  "gpu": "Tesla T4",
  "gpu_vram_gb": 15.6,
  "cuda": "12.8"
}

=== 5. base.en + sortformer ===


diar_sortformer_4spk-v1.nemo: reconstructing file:   0%|          |  0.00B /  493MB            

diar_sortformer_4spk-v1.nemo: downloading bytes:           |  0.00B            

[NeMo W 2026-08-19 15:11:32 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 4
    shuffle: true
    num_workers: 18
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    num_buckets: 10
    bucket_duration_bins:
    - 10
    - 20
    - 30
    - 40
    - 50
    - 60
    - 70
    - 80
    - 90
    pin_memory: true
    min_duration: 80
    max_duration: 90
    batch_duration: 400
    quadratic_duration: 1200
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-08-19 15:11:32 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or

[NeMo I 2026-08-19 15:11:34 save_restore_connector:287] Model SortformerEncLabelModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--diar_sortformer_4spk-v1/snapshots/9f17b10df44c0a4c8f3c86fbddc9ee2d6ab9ac08/diar_sortformer_4spk-v1.nemo.
  load 9.51s, VRAM 1.01GB
[NeMo I 2026-08-19 15:11:34 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


...

  script 2: word 94.8%  speaker 99.5%  15/17 turns  [saved]
[NeMo I 2026-08-19 15:11:38 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


[NeMo W 2026-08-19 15:11:38 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: soft_label_thres,session_len_sec,num_spks
Diarizing: 1it [00:00,  2.76it/s]


  script 3: word 87.8%  speaker 100.0%  16/17 turns  [saved]
[NeMo I 2026-08-19 15:11:39 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


[NeMo W 2026-08-19 15:11:39 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: soft_label_thres,session_len_sec,num_spks
Diarizing: 1it [00:00,  1.29it/s]

  script 4: word 83.6%  speaker 100.0%  14/16 turns  [saved]

MEAN  word 88.3%   speaker 99.9%   infer 6.3s   peak VRAM 1.26GB   peak RAM 2.31GB

Expected from the earlier runs: word ~88%, speaker ~99.9%, VRAM ~3GB.
If it lands far from that, tell me before it goes in the report.

Download base_sortformer_detail.csv and send it to me.
